In [ ]:
import asyncio
import json
import os
import time

from dotenv import load_dotenv
from agents import Agent, Runner, trace

load_dotenv(override=True)

# 이 과제에서 쓰는 키는 OPENAI_API_KEY 하나뿐이다.
openai_api_key = os.getenv("OPENAI_API_KEY")
print("OpenAI 키:", f"있음 ({openai_api_key[:8]}...)" if openai_api_key else "없음")

# 세 전문가와 관리자 모두 이 모델 하나를 쓴다. 바꾸는 변수를 하나로 둬야 비교가 성립한다.
MODEL_NAME = "gpt-5.4-mini"

# 방식마다 반복 횟수 (3회 이상)
REPEATS = 3

print("모델:", MODEL_NAME, "/ 반복:", REPEATS)

In [ ]:
def load_assets():
    """자산 인벤토리를 읽어옵니다."""
    with open(ASSETS_PATH, encoding="utf-8") as f:
        return json.load(f)


def search_cves_for_product(product: str, days: int = 120, max_results: int = 4):
    """특정 제품과 관련된 최근 CVE를 NVD에서 조회합니다.

    Args:
        product: 제품 이름 (예: Apache Tomcat)
        days: 최근 며칠 이내에 공개된 것을 볼지
        max_results: 제품당 최대 건수
    """
    end = datetime.now(timezone.utc)
    start = end - timedelta(days=days)

    # 키가 있으면 apiKey 헤더로 보낸다. 없으면 익명 호출(제한이 더 빡빡함).
    headers = {"apiKey": NVD_KEY} if NVD_KEY else {}

    r = requests.get(
        "https://services.nvd.nist.gov/rest/json/cves/2.0",
        params={
            "keywordSearch": product,
            "pubStartDate": start.strftime("%Y-%m-%dT%H:%M:%S.000"),
            "pubEndDate": end.strftime("%Y-%m-%dT%H:%M:%S.000"),
            "resultsPerPage": max_results,
            "noRejected": None,
        },
        headers=headers,
        timeout=30,
    )
    r.raise_for_status()

    result = []
    for item in r.json().get("vulnerabilities", []):
        cve = item["cve"]
        metrics = cve.get("metrics", {})
        # CVSS 버전이 여러 개일 수 있으므로 최신 것부터 찾는다
        key = next((k for k in ["cvssMetricV31", "cvssMetricV40", "cvssMetricV30", "cvssMetricV2"]
                    if k in metrics), None)
        cvss = metrics[key][0]["cvssData"] if key else {}

        result.append({
            "cve_id": cve["id"],
            "published": cve["published"][:10],
            "cvss_score": float(cvss.get("baseScore", 0.0)),
            "severity": cvss.get("baseSeverity", "UNKNOWN"),
            "description": cve["descriptions"][0]["value"][:400],
        })
    return result

In [ ]:
class MatchedCVE(BaseModel):
    cve_id: str
    asset_id: str
    reason: str          # 왜 영향을 준다고 판단했는지
    cvss_score: float
    summary_ko: str      # 취약점 내용 한글 요약


class ExcludedCVE(BaseModel):
    cve_id: str
    reason: str          # 왜 제외했는지


class MatchResult(BaseModel):
    matched: list[MatchedCVE]
    excluded: list[ExcludedCVE]


matcher = Agent(
    name="자산 매칭 에이전트",
    model=MODEL_NAME,
    instructions=(
        "당신은 취약점 관리 담당자입니다. 자산 목록과 CVE 후보 목록을 받습니다.\n"
        "각 CVE가 해당 자산에 '실제로' 영향을 주는지 판단하세요.\n"
        "- 키워드만 겹치고 다른 제품이면 제외하세요. "
        "(예: 자산은 OpenSSL인데 CVE는 rust-openssl 바인딩 취약점)\n"
        "- 설명에 명시된 영향 버전 범위가 자산 버전을 포함하지 않으면 제외하세요.\n"
        "- 버전 정보가 불명확하면 matched 에 넣되 reason에 '버전 확인 필요'라고 적으세요.\n"
        "제외한 것은 excluded 에 이유와 함께 넣으세요. reason은 한 문장으로 쓰세요."
    ),
    output_type=MatchResult,
)


class Advisory(BaseModel):
    cve_id: str
    asset_id: str
    asset_name: str
    severity: str        # 긴급 / 높음 / 보통 / 낮음
    summary_ko: str
    action: str          # 담당자가 바로 실행할 조치
    deadline: str        # 권고 기한


class OwnerBriefing(BaseModel):
    owner: str
    owner_email: str
    headline: str        # 메일 제목
    advisories: list[Advisory]


class BriefingResult(BaseModel):
    briefings: list[OwnerBriefing]


briefer = Agent(
    name="브리핑 작성 에이전트",
    model=MODEL_NAME,
    instructions=(
        "당신은 보안 어드바이저리 담당자입니다. 확정된 취약점 목록과 자산 정보를 받습니다.\n"
        "위험도(severity)는 CVSS 점수만이 아니라 자산의 중요도와 인터넷 노출 여부를 함께 고려해 "
        "'긴급' / '높음' / '보통' / '낮음' 중 하나로 정하세요.\n"
        "- 인터넷에 노출되고 중요도가 '상'인 자산은 한 단계 올립니다\n"
        "- 내부망 전용이면 한 단계 내릴 수 있습니다\n"
        "action에는 담당자가 바로 실행할 수 있는 구체적 조치를 1~2문장으로 쓰세요.\n"
        "deadline은 '24시간 이내', '1주일 이내' 처럼 적으세요.\n"
        "담당자(owner)별로 묶어서 출력하고, headline은 그 담당자에게 보낼 메일 제목입니다."
    ),
    output_type=BriefingResult,
)

In [ ]:
SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

# 데모 안전장치 1: 모든 메일을 본인 주소로 보냄. 실무에서는 None 으로 바꾼다.
ROUTE_ALL_TO_ME = EMAIL_ADDRESS

# 데모 안전장치 2: True 면 실제로 보내지 않고 "보냈다고 가정"만 한다.
DRY_RUN = False

SEVERITY_COLOR = {"긴급": "#c0392b", "높음": "#e67e22", "보통": "#2980b9", "낮음": "#7f8c8d"}


def build_html(briefing: OwnerBriefing) -> str:
    """담당자 한 명에게 보낼 HTML 메일 본문을 만듭니다."""
    rows = ""
    for a in briefing.advisories:
        color = SEVERITY_COLOR.get(a.severity, "#7f8c8d")
        rows += f"""
        <tr>
          <td style="padding:8px;border-bottom:1px solid #eee;">
            <span style="background:{color};color:#fff;padding:2px 8px;border-radius:3px;font-size:12px;">{a.severity}</span>
          </td>
          <td style="padding:8px;border-bottom:1px solid #eee;font-family:monospace;">{a.cve_id}</td>
          <td style="padding:8px;border-bottom:1px solid #eee;">{a.asset_name}</td>
          <td style="padding:8px;border-bottom:1px solid #eee;">{a.deadline}</td>
        </tr>
        <tr>
          <td></td>
          <td colspan="3" style="padding:0 8px 12px 8px;border-bottom:1px solid #eee;color:#555;font-size:13px;">
            {a.summary_ko}<br><b>조치:</b> {a.action}
          </td>
        </tr>"""

    return f"""
    <div style="font-family:'Malgun Gothic',sans-serif;max-width:720px;">
      <h2 style="border-bottom:2px solid #333;padding-bottom:8px;">보안 취약점 일일 브리핑</h2>
      <p>{briefing.owner}님, 담당 자산에서 조치가 필요한 취약점
         <b>{len(briefing.advisories)}건</b>이 확인되었습니다.</p>
      <table style="border-collapse:collapse;width:100%;font-size:14px;">
        <tr style="background:#f4f4f4;">
          <th style="padding:8px;text-align:left;">위험도</th>
          <th style="padding:8px;text-align:left;">CVE</th>
          <th style="padding:8px;text-align:left;">자산</th>
          <th style="padding:8px;text-align:left;">조치 기한</th>
        </tr>
        {rows}
      </table>
      <p style="color:#888;font-size:12px;margin-top:16px;">
        본 메일은 보안 브리핑 매니저 에이전트가 NVD 데이터를 기반으로 자동 생성했습니다. (수업용 데모)
      </p>
    </div>"""


def send_email(to_address: str, subject: str, html_body: str) -> str:
    """HTML 메일을 발송합니다."""
    msg = EmailMessage()
    msg["Subject"] = subject
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = to_address
    msg.set_content("HTML 메일입니다. HTML을 지원하는 클라이언트에서 확인하세요.")
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP_SSL(SMTP_SERVER, 465) as smtp:
        smtp.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        smtp.send_message(msg)

    return f"발송 완료 -> {to_address}"

In [ ]:
# 작업 메모리: 도구들이 주고받는 큰 데이터를 보관한다. 매니저(LLM)는 이 내용을 직접 보지 않는다.
state = {}


def reset_state():
    """새 실행을 시작할 때 작업 메모리를 비웁니다."""
    state.clear()
    state.update({
        "assets": [],        # list_assets 결과
        "candidates": {},    # 제품 -> CVE 후보 목록 (search_cves)
        "matched": [],       # 실제 영향 CVE (match_cves)
        "briefings": {},     # 담당자 -> OwnerBriefing (write_briefings)
        "sent": set(),       # 이미 메일을 보낸 담당자 (중복 발송 방지)
        "skipped": {},       # 보내지 않기로 한 담당자 -> 이유 (6단계)
    })


reset_state()


# 매니저가 search_cves 를 동시에 여러 번 부를 수 있으므로, NVD 호출 간격을 코드로 지킨다.
_nvd_lock = threading.Lock()
_nvd_last_call = 0.0


def nvd_throttle():
    """직전 NVD 호출로부터 NVD_DELAY 초가 지날 때까지 기다립니다."""
    global _nvd_last_call
    with _nvd_lock:
        wait = NVD_DELAY - (time.monotonic() - _nvd_last_call)
        if wait > 0:
            time.sleep(wait)
        _nvd_last_call = time.monotonic()

In [ ]:
def list_assets() -> str:
    """회사 자산 목록을 불러옵니다. 작업을 시작할 때 가장 먼저 호출하세요.
    각 자산의 id, 이름, 제품명(product), 버전, 담당자(owner), 중요도, 인터넷 노출 여부를 돌려줍니다.
    """
    assets = load_assets()["assets"]
    state["assets"] = assets

    summary = [
        {k: a[k] for k in ("id", "name", "product", "version", "owner", "criticality", "internet_facing")}
        for a in assets
    ]
    return json.dumps(summary, ensure_ascii=False)


def search_cves(product: str) -> str:
    """NVD에서 제품 하나의 최근 CVE 후보를 조회해 작업 메모리에 저장합니다.
    조회할 제품마다 한 번씩 호출하세요. 여러 제품을 동시에 호출해도 됩니다.

    Args:
        product: 조회할 제품 이름. list_assets 결과의 product 값을 그대로 쓰세요. (예: Apache Tomcat)
    """
    nvd_throttle()
    try:
        cves = search_cves_for_product(product)
    except requests.RequestException as e:
        return f"{product}: NVD 조회 실패 ({type(e).__name__}). 잠시 후 다시 호출해 보세요."

    state["candidates"][product] = cves
    if not cves:
        return f"{product}: 최근 CVE 후보 없음"
    ids = ", ".join(f"{c['cve_id']}(CVSS {c['cvss_score']})" for c in cves)
    return f"{product}: 후보 {len(cves)}건 저장 — {ids}"

In [ ]:
print(list_assets()[:300], "...\n")
print(search_cves("Apache Tomcat"))

reset_state()   # 테스트로 채운 작업 메모리를 비운다

In [ ]:
async def match_cves() -> str:
    """수집한 CVE 후보 중 우리 자산에 실제로 영향을 주는 것만 골라냅니다. (자산 매칭 에이전트 호출)
    조회할 제품의 search_cves 가 모두 끝난 뒤 한 번 호출하세요.
    """
    if not state["assets"]:
        return "자산 목록이 없습니다. 먼저 list_assets 를 호출하세요."
    if not state["candidates"]:
        return "CVE 후보가 없습니다. 먼저 search_cves 로 후보를 모으세요."

    payload = {
        "assets": [
            {k: a[k] for k in ("id", "name", "product", "version", "criticality", "internet_facing")}
            for a in state["assets"]
        ],
        "cve_candidates": state["candidates"],
    }
    m = await Runner.run(matcher, json.dumps(payload, ensure_ascii=False))

    result = m.final_output
    state["matched"] = result.matched
    state["briefings"] = {}      # 매칭이 바뀌면 이전 브리핑은 무효

    total = sum(len(v) for v in state["candidates"].values())
    head = (f"제품 {len(state['candidates'])}종, 후보 {total}건 중 "
            f"실제 영향 {len(result.matched)}건, 제외 {len(result.excluded)}건")
    if not result.matched:
        return head + " — 영향받는 취약점이 없으므로 브리핑과 메일은 필요 없습니다."
    lines = [f"- {x.cve_id} -> {x.asset_id} (CVSS {x.cvss_score}): {x.reason}" for x in result.matched]
    return head + "\n" + "\n".join(lines)


async def write_briefings() -> str:
    """확정된 취약점으로 담당자별 브리핑(위험도, 조치, 기한, 메일 제목)을 작성합니다. (브리핑 작성 에이전트 호출)
    match_cves 가 끝난 뒤 호출하세요.
    """
    if not state["matched"]:
        return "확정된 취약점이 없습니다. match_cves 를 먼저 호출했는지 확인하세요."

    asset_info = {
        a["id"]: {
            "owner": a["owner"],
            "owner_email": a["owner_email"],
            "name": a["name"],
            "criticality": a["criticality"],
            "internet_facing": a["internet_facing"],
        }
        for a in state["assets"]
    }
    b = await Runner.run(
        briefer,
        json.dumps({"matched": [x.model_dump() for x in state["matched"]], "asset_info": asset_info},
                   ensure_ascii=False),
    )

    state["briefings"] = {br.owner: br for br in b.final_output.briefings}

    lines = []
    for br in state["briefings"].values():
        counts = Counter(a.severity for a in br.advisories)
        levels = ", ".join(f"{s} {counts[s]}" for s in SEVERITY_COLOR if counts[s])
        lines.append(f"- {br.owner}: {len(br.advisories)}건 ({levels}) / 제목: {br.headline}")
    return f"담당자 {len(state['briefings'])}명의 브리핑 작성 완료\n" + "\n".join(lines)

In [ ]:
def send_briefing_email(owner: str) -> str:
    """담당자 한 명에게 write_briefings 로 작성한 브리핑 메일을 보냅니다.
    받는 주소와 메일 본문은 작성된 브리핑으로 코드가 만들므로, 담당자 이름만 넘기면 됩니다.
    담당자 한 명당 한 번만 호출하세요.

    Args:
        owner: 담당자 이름. write_briefings 결과에 나온 이름 그대로 쓰세요. (예: 김운영)
    """
    br = state["briefings"].get(owner)
    if br is None:
        available = ", ".join(state["briefings"]) or "없음 (write_briefings 를 먼저 호출하세요)"
        return f"'{owner}' 의 브리핑이 없습니다. 보낼 수 있는 담당자: {available}"
    if owner in state["sent"]:
        return f"{owner} 에게는 이미 발송했습니다. 다시 보내지 않습니다."

    to = ROUTE_ALL_TO_ME or br.owner_email
    if DRY_RUN:
        result = f"[DRY_RUN] 발송 생략 -> {to} / 제목: {br.headline}"
    else:
        result = send_email(to, br.headline, build_html(br))

    state["sent"].add(owner)
    return f"{owner}: {result}"

In [ ]:
tools = [
    function_tool(list_assets),
    function_tool(search_cves),
    function_tool(match_cves),
    function_tool(write_briefings),
    function_tool(send_briefing_email),
]

for t in tools:
    print(f"{t.name:<20} {t.description.splitlines()[0]}")

In [ ]:
# 매니저(LLM)가 보는 search_cves 도구의 입력 스키마
tools[1].params_json_schema

In [ ]:
instructions = """
당신은 한빛테크 주식회사의 보안 브리핑 매니저입니다.
도구를 사용해 오늘의 보안 취약점 브리핑을 담당자에게 전달하는 것이 목표입니다.

규칙:
- 큰 데이터는 도구가 작업 메모리에 저장합니다. 당신은 도구가 돌려주는 요약을 보고 다음에 할 일을 결정합니다.
- CVE, 위험도, 담당자 정보를 스스로 지어내지 마세요. 도구 결과만 근거로 삼으세요.
- 도구가 실패했거나 선행 단계가 필요하다고 알려 주면, 그 안내에 따라 대응하세요.
- 같은 담당자에게 메일을 두 번 보내지 마세요.

작업이 끝나면 무엇을 했는지 한국어로 짧게 보고하세요.
(조회한 제품 수, 확정된 취약점 수, 메일을 보낸 담당자와 결과, 보내지 않은 담당자가 있다면 그 이유)
"""

task = """
오늘의 보안 브리핑을 진행하세요.

1. 자산 확인: list_assets 로 자산 목록을 불러오세요.
2. CVE 수집: 자산에 있는 모든 제품에 대해 search_cves 를 호출하세요. 제품마다 한 번씩, 빠짐없이.
3. 자산 매칭: 모든 제품의 수집이 끝난 뒤 match_cves 를 한 번 호출하세요.
4. 브리핑 작성: 확정된 취약점이 있으면 write_briefings 를 호출하세요.
5. 발송: 브리핑이 작성된 담당자마다 send_briefing_email 을 한 번씩 호출하세요.
"""

security_manager = Agent(
    name="보안 브리핑 매니저",
    instructions=instructions,
    tools=tools,
    model=MODEL_NAME,
)

In [ ]:
reset_state()

with trace("보안 브리핑 매니저"):
    result = await Runner.run(security_manager, task, max_turns=30)

print(result.final_output)

In [ ]:
def show_run(result):
    """에이전트가 어떤 도구를 어떤 순서로 불렀는지 출력합니다. (핸드오프 포함)"""
    outputs = {}
    for item in result.new_items:
        if item.type == "tool_call_output_item":
            raw = item.raw_item
            call_id = raw["call_id"] if isinstance(raw, dict) else raw.call_id
            outputs[call_id] = str(item.output)

    calls = ("tool_call_item", "handoff_call_item")
    decision, prev_type, n_calls = 0, None, 0
    for item in result.new_items:
        if item.type in calls:
            if prev_type not in calls:             # 연속된 호출 = 한 번의 판단에서 병렬 호출
                decision += 1
                print(f"--- 판단 {decision}: {item.agent.name} ---")
            call = item.raw_item
            args = "" if call.arguments in ("", "{}") else call.arguments
            print(f"  {call.name}({args})")
            if item.type == "tool_call_item":
                n_calls += 1
                out = outputs.get(call.call_id, "")
                print(f"      -> {out.splitlines()[0][:90] if out else '(결과 없음)'}")
        elif item.type == "handoff_output_item":
            print(f"      -> 핸드오프: {item.source_agent.name} → {item.target_agent.name} (제어가 넘어감)")
        prev_type = item.type

    print(f"\nLLM 호출 {len(result.raw_responses)}회 (매칭·브리핑 에이전트 제외) / 도구 호출 {n_calls}회 / "
          f"마지막 에이전트: {result.last_agent.name}")


show_run(result)

In [ ]:
print(f"확정 취약점 {len(state['matched'])}건 / 브리핑 {len(state['briefings'])}명 / "
      f"발송 완료: {', '.join(sorted(state['sent'])) or '없음'}\n")

for br in state["briefings"].values():
    print(f"=== {br.owner} <{br.owner_email}> — {len(br.advisories)}건")
    print(f"    제목: {br.headline}")
    for a in br.advisories:
        print(f"    [{a.severity:<3}] {a.cve_id:<16} {a.asset_name:<18} {a.deadline}")
    print()

In [ ]:
task2 = """
오늘은 인터넷에 노출된 자산만 점검하세요.

1. list_assets 로 자산을 확인하고, internet_facing 이 true 인 자산의 제품만 search_cves 로 조회하세요.
2. match_cves, write_briefings 로 브리핑을 만드세요.
3. '긴급' 건이 있는 담당자에게만 send_briefing_email 을 호출하세요.
   메일을 보내지 않은 담당자가 있으면 보고에 그 이유를 적으세요.
"""

prev_dry_run, DRY_RUN = DRY_RUN, True     # 이번 실행은 판단만 보고 실제로는 보내지 않는다
reset_state()
try:
    with trace("보안 브리핑 매니저 - 인터넷 노출 자산만"):
        result2 = await Runner.run(security_manager, task2, max_turns=30)
finally:
    DRY_RUN = prev_dry_run                 # 원래 설정으로 되돌린다

show_run(result2)
print("\n[ 매니저 보고 ]")
print(result2.final_output)

In [ ]:
def skip_briefing_email(owner: str, reason: str) -> str:
    """발송 조건에 맞지 않아 이 담당자에게는 메일을 보내지 않기로 한 결정을 기록합니다.

    Args:
        owner: 담당자 이름. write_briefings 결과에 나온 이름 그대로 쓰세요. (예: 박데이터)
        reason: 보내지 않는 이유 한 문장 (예: 긴급·높음 건이 없음)
    """
    if owner not in state["briefings"]:
        available = ", ".join(state["briefings"]) or "없음"
        return f"'{owner}' 의 브리핑이 없습니다. 담당자: {available}"
    if owner in state["sent"]:
        return f"{owner} 에게는 이미 발송했습니다. 미발송으로 바꿀 수 없습니다."

    state["skipped"][owner] = reason
    return f"{owner}: 발송 안 함 — {reason}"


mail_sender = Agent(
    name="메일 발송 에이전트",
    handoff_description="담당자별 브리핑 작성이 끝난 뒤, 메일 발송을 맡는 에이전트",
    instructions=RECOMMENDED_PROMPT_PREFIX + """
당신은 보안 브리핑 메일 발송 담당자입니다. 보안 분석 매니저에게서 업무를 넘겨받습니다.

- 대화 기록에서 write_briefings 결과를 찾아, 브리핑이 작성된 담당자와 위험도별 건수를 확인하세요.
  위험도는 메일 제목이 아니라 괄호 안의 위험도별 건수로 판단하세요.
- 사용자의 요청에 발송 조건이 있으면 그 조건을 따르세요.
- 브리핑이 작성된 담당자 '모두'에 대해 둘 중 하나를 반드시 호출하세요.
  - 조건에 맞으면 send_briefing_email (메일 주소와 본문은 도구가 만듭니다)
  - 조건에 맞지 않으면 skip_briefing_email (reason 에는 그 담당자의 위험도별 건수를 함께 적으세요)
- 도구를 호출하지 않고 발송했다고 보고하면 안 됩니다. 보고는 도구 결과만 근거로 하세요.

끝나면 한국어로 짧게 보고하세요. (발송한 담당자와 결과, 보내지 않은 담당자와 그 이유)
""",
    tools=[function_tool(send_briefing_email), function_tool(skip_briefing_email)],
    model=MODEL_NAME,
    model_settings=ModelSettings(tool_choice="required"),   # 첫 응답에서 반드시 도구를 호출
)

# 한글 에이전트 이름으로는 도구 이름이 깨지므로 영문 이름을 직접 준다
to_mail_sender = handoff(mail_sender, tool_name_override="transfer_to_mail_sender")

print("핸드오프 도구 이름:", to_mail_sender.tool_name)
print("핸드오프 도구 설명:", to_mail_sender.tool_description)

In [ ]:
analysis_tools = [t for t in tools if t.name != "send_briefing_email"]

analysis_instructions = RECOMMENDED_PROMPT_PREFIX + """
당신은 한빛테크 주식회사의 보안 분석 매니저입니다.
도구로 취약점을 분석해 담당자별 브리핑을 작성한 뒤, 메일 발송은 메일 발송 에이전트에게 넘깁니다.
당신에게는 메일 발송 도구가 없습니다.

규칙:
- 큰 데이터는 도구가 작업 메모리에 저장합니다. 당신은 도구가 돌려주는 요약을 보고 다음에 할 일을 결정합니다.
- CVE, 위험도, 담당자 정보를 스스로 지어내지 마세요. 도구 결과만 근거로 삼으세요.
- 도구가 실패했거나 선행 단계가 필요하다고 알려 주면, 그 안내에 따라 대응하세요.
- 확정된 취약점이 없으면 넘기지 말고, 그 사실을 보고하고 끝내세요.
"""

handoff_task = """
오늘의 보안 브리핑을 진행하세요.

1. 자산 확인: list_assets 로 자산 목록을 불러오세요.
2. CVE 수집: 자산에 있는 모든 제품에 대해 search_cves 를 호출하세요. 제품마다 한 번씩, 빠짐없이.
3. 자산 매칭: 모든 제품의 수집이 끝난 뒤 match_cves 를 한 번 호출하세요.
4. 브리핑 작성: 확정된 취약점이 있으면 write_briefings 를 호출하세요.
5. 발송: 메일 발송 에이전트에게 넘기세요.
   발송 조건은 '긴급' 또는 '높음' 건이 하나라도 있는 담당자에게만 보내는 것입니다.
"""

analysis_manager = Agent(
    name="보안 분석 매니저",
    instructions=analysis_instructions,
    tools=analysis_tools,
    handoffs=[to_mail_sender],
    model=MODEL_NAME,
)

print("분석 매니저가 보는 도구:", [t.name for t in analysis_tools] + [to_mail_sender.tool_name])

In [ ]:
prev_dry_run, DRY_RUN = DRY_RUN, True     # 4단계에서 이미 보냈으므로 이번에는 판단만 본다
reset_state()
try:
    with trace("보안 브리핑 - 핸드오프"):
        result3 = await Runner.run(analysis_manager, handoff_task, max_turns=30)
finally:
    DRY_RUN = prev_dry_run                 # 원래 설정으로 되돌린다

show_run(result3)
print(f"\n[ {result3.last_agent.name} 의 최종 보고 ]")
print(result3.final_output)

In [ ]:
handed_off = result3.last_agent is mail_sender
decided = state["sent"] | set(state["skipped"])
undecided = sorted(set(state["briefings"]) - decided)

print(f"핸드오프: {'됨' if handed_off else '안 됨 — 매니저가 넘기지 않고 끝냈습니다'}")
print(f"브리핑 {len(state['briefings'])}명 / 발송 {len(state['sent'])}명: "
      f"{', '.join(sorted(state['sent'])) or '없음'}")
for owner, reason in state["skipped"].items():
    print(f"  발송 안 함: {owner} — {reason}")
if undecided:
    print(f"기록 없는 담당자: {', '.join(undecided)} — 에이전트가 도구를 부르지 않았습니다. "
          f"보고를 믿지 말고 다시 실행하세요.")

# 발송 조건('긴급' 또는 '높음' 건이 하나라도 있음)을 코드로 다시 계산해 LLM의 결정과 비교한다
print("\n[ 발송 조건 검사 ]")
for owner, br in state["briefings"].items():
    should_send = any(a.severity in ("긴급", "높음") for a in br.advisories)
    did_send = owner in state["sent"]
    verdict = "OK" if should_send == did_send else "불일치"
    print(f"  [{verdict}] {owner}: 조건 {'충족' if should_send else '미충족'} → "
          f"{'발송' if did_send else '발송 안 함'}")